# Notebook 4: Roofline Model
Based on Williams et al. (2009)
Plots roofline for A100 GPU with LLM attention operations.

In [ ]:
!pip install matplotlib numpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("results/figures", exist_ok=True)

PEAK_FLOPS_TFLOPS = 312
PEAK_BW_TBS = 2.0

ai = np.logspace(-2, 4, 1000)
perf_roof = np.minimum(PEAK_FLOPS_TFLOPS, PEAK_BW_TBS * ai)

operations = [
    ("Naive Attn FP16", 2*512*512*64, 3*512*64*2),
    ("Attn INT8",       2*512*512*64, 3*512*64*1),
    ("Attn INT4",       2*512*512*64, 3*512*64*0.5),
    ("FlashAttention",  2*512*512*64, 512*64*2 + 512*64*2),
    ("FFN Layer",       2*512*4096*1024, 3*512*1024*2),
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ai, perf_roof, 'b-', linewidth=2.5, label='A100 Roofline')
ax.axvline(x=PEAK_FLOPS_TFLOPS/PEAK_BW_TBS, color='blue',
           linestyle='--', alpha=0.5, label='Ridge Point (156 FLOP/byte)')

colors = ['#e74c3c','#e67e22','#f39c12','#2ecc71','#9b59b6']
markers = ['o','s','^','D','v']

for (name, flops, bytes_accessed), color, marker in zip(operations, colors, markers):
    ai_val = flops / bytes_accessed
    achievable = min(PEAK_FLOPS_TFLOPS, PEAK_BW_TBS * ai_val)
    ax.scatter(ai_val, achievable, s=120, color=color, marker=marker, zorder=5, label=name)
    ax.annotate(name, (ai_val, achievable), textcoords="offset points", xytext=(8,5), fontsize=9)

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOP/s)', fontsize=12)
ax.set_title('Roofline Model — A100 GPU (Williams et al., 2009)', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/roofline_model.png", dpi=150)
print("Saved roofline_model.png")
plt.show()